# FinMiniLM Fine-Tune — Contrastive Learning on PDF-Domain Pairs

## Goal
Fine-tune `all-MiniLM-L6-v2` on (question, S4_chunk) pairs from your actual PDFs.
Uses **MultipleNegativesRankingLoss** — industry standard for bi-encoder retrieval fine-tuning.

## Data
`fine_tuning/training_pairs.csv` — ~2,400 (question, chunk) pairs across 4 categories.
All questions generated by Groq FROM your 41 PDFs. All chunks are S4 token-exact.

## Saved to
`fine_tuning/minilm_finetuned/` — load with `SentenceTransformer('fine_tuning/minilm_finetuned')`

In [ ]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

FINE_TUNE_DIR = 'fine_tuning'
PAIRS_PATH    = os.path.join(FINE_TUNE_DIR, 'training_pairs.csv')
MODEL_SAVE    = os.path.join(FINE_TUNE_DIR, 'minilm_finetuned')
RESULTS_DIR   = os.path.join(FINE_TUNE_DIR, 'Results')
MINILM_NAME   = 'sentence-transformers/all-MiniLM-L6-v2'

os.makedirs(MODEL_SAVE,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(PAIRS_PATH), 'Run FinS4_BuildCorpus.ipynb first!'
print('Config OK')
print('Training pairs:', PAIRS_PATH)
print('Model save    :', MODEL_SAVE)

In [ ]:
import sys
!{sys.executable} -m pip install -q sentence-transformers torch datasets

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

print('Imports OK')
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## Cell 3 — Load Training Pairs

In [ ]:
pairs_df = pd.read_csv(PAIRS_PATH)
print('Training pairs loaded: {:,}'.format(len(pairs_df)))
print('Category distribution:')
print(pairs_df['category'].value_counts().to_string())
print('\nAvg similarity score : {:.4f}'.format(pairs_df['sim_score'].mean()))
print('Min similarity score : {:.4f}'.format(pairs_df['sim_score'].min()))
print('\nSample pair:')
sample = pairs_df.iloc[0]
print('  Q:', sample['question'][:80])
print('  C:', sample['chunk_text'][:80])
print('  Sim:', sample['sim_score'])

# Build InputExamples — (question, chunk) pairs
train_examples = [
    InputExample(texts=[row['question'], row['chunk_text']])
    for _, row in pairs_df.iterrows()
]

BATCH_SIZE       = 32
train_dataloader = DataLoader(train_examples, shuffle=True,
                               batch_size=BATCH_SIZE, drop_last=True)

print('\nDataLoader:')
print('  Pairs        : {:,}'.format(len(train_examples)))
print('  Batch size   : {}'.format(BATCH_SIZE))
print('  Batches/epoch: {:,}'.format(len(train_dataloader)))

## Cell 4 — Fine-Tune MiniLM (~30-40 min on CPU)

In [ ]:
EPOCHS = 3
WARMUP = int(len(train_dataloader) * 0.1)

print('Loading base MiniLM...')
model      = SentenceTransformer(MINILM_NAME)
train_loss = losses.MultipleNegativesRankingLoss(model)

print('\nStarting fine-tuning...')
print('  Pairs        : {:,}'.format(len(train_examples)))
print('  Epochs       : {}'.format(EPOCHS))
print('  Batch size   : {}'.format(BATCH_SIZE))
print('  Warmup steps : {}'.format(WARMUP))
print('  Loss         : MultipleNegativesRankingLoss')
print('  Expected time: 30-40 min on CPU')
print()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    output_path=MODEL_SAVE,
    save_best_model=True,
    show_progress_bar=True,
)

print('\nFine-tuning complete.')
print('Model saved to:', MODEL_SAVE)

In [ ]:
summary = {
    'base_model':     MINILM_NAME,
    'saved_to':       MODEL_SAVE,
    'training_pairs': len(train_examples),
    'epochs':         EPOCHS,
    'batch_size':     BATCH_SIZE,
    'loss':           'MultipleNegativesRankingLoss',
    'data_source':    'Groq-generated questions mapped to S4 chunks from 41 PDFs',
}
with open(os.path.join(RESULTS_DIR, 'finetune_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved.')
print()
print('Next: run FinMiniLM_Eval.ipynb to compare base vs fine-tuned on synthetic QA pairs.')